# QPY Transpiled Circuit Statistics

This notebook scans the cached transpiled QPY circuits from the hardware QRC run and computes per-circuit gate/depth statistics.

Default cache path:

`Classical/results/hardware_qrc_run/transpile_cache`

Notes:

- The cache is read-only from this notebook; generated summaries are written to `Classical/results/hardware_qrc_run/qpy_transpile_stats/`.
- The run used `reuse-z-basis`, so the cache contains transpiled Z-basis circuits. X/Y basis circuits were derived in memory during submission.
- The cache may include legacy cache-key copies. Per-circuit stats are valid; aggregate total gate counts may double-count equivalent cached circuits if duplicates are present.
- Gate stats and runtime estimates use the first `GATE_FRACTION` of non-measure gates from each cached circuit (default `0.5`). Set `GATE_FRACTION = 1.0` to analyze full circuits.
- A later section estimates IBM Runtime quantum time from cached Z circuits using `FakeFez` scheduling (`estimate_duration`), and compares that to the coarse `hardware_qrc.py` offline formula.

In [15]:
from __future__ import annotations

import json
import math
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from IPython.display import display
from qiskit import qpy

# Resolve paths whether the notebook is launched from repo root or Classical/.
cwd = Path.cwd().resolve()
if (cwd / "Classical").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "Classical":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CACHE_DIR = PROJECT_ROOT / "results/NN-TFI/transpile_cache"
OUTPUT_DIR = PROJECT_ROOT / "results/NN-TFI/qpy_transpile_stats"

# Set MAX_FILES to a small integer for a smoke test, or None for the full cache.
MAX_FILES = None

# Use the first GATE_FRACTION of non-measure gates when computing stats/durations.
# Set to 1.0 to analyze full transpiled circuits.
GATE_FRACTION = 0.5

# Threading avoids multiprocessing pickling issues in notebooks and still overlaps QPY I/O.
MAX_WORKERS = 6
PROGRESS_EVERY = 500
SAVE_OUTPUTS = True

# Hardware runtime estimate (offline FakeFez target; mirrors Classical/hardware_qrc.py).
BACKEND_NAME = "ibm_fez"
SHOTS = 4096
PUBS_PER_JOB = 500
TRANSPILE_STRATEGY = "reuse-z-basis"  # "reuse-z-basis" | "independent"
PER_SUB_JOB_OVERHEAD_SECONDS = 2.0
REP_DELAY_SECONDS = 250e-6
SCHEDULING_OPTIMIZATION_LEVEL = 0
QUICK_SECONDS_PER_EXECUTION = 0.00035
# Set to a small integer for a smoke test, or None to estimate every cached circuit.
DURATION_MAX_FILES = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

qpy_files = sorted(CACHE_DIR.glob("*.qpy"))
if MAX_FILES is not None:
    qpy_files = qpy_files[:MAX_FILES]

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache dir:    {CACHE_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Gate fraction:{GATE_FRACTION:.0%} of non-measure gates")
print(f"QPY files:    {len(qpy_files):,}")
if not qpy_files:
    raise FileNotFoundError(f"No .qpy files found in {CACHE_DIR}")

Project root: /Users/noah/repos/Icequake-QRC-/Classical
Cache dir:    /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/transpile_cache
Output dir:   /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats
QPY files:    7,056


In [17]:
from qiskit import QuantumCircuit


def subset_circuit_gates(circuit: QuantumCircuit, fraction: float) -> QuantumCircuit:
    """Keep the first fraction of non-measure gates; retain all measurements."""
    if fraction >= 1.0:
        return circuit

    non_measure = [inst for inst in circuit.data if inst.operation.name != "measure"]
    measures = [inst for inst in circuit.data if inst.operation.name == "measure"]
    if not non_measure:
        return circuit

    keep = max(1, int(len(non_measure) * fraction))
    subset = QuantumCircuit(circuit.num_qubits, circuit.num_clbits, name=circuit.name)
    for inst in non_measure[:keep] + measures:
        subset.append(inst.operation, inst.qubits, inst.clbits)
    return subset


def inspect_qpy_file(path: Path, gate_fraction: float = GATE_FRACTION) -> dict:
    """Load one QPY file and return per-circuit stats.

    Each cached file is expected to contain exactly one transpiled circuit.
    """
    with path.open("rb") as f:
        circuits = qpy.load(f)

    if len(circuits) != 1:
        raise ValueError(f"{path.name}: expected 1 circuit, got {len(circuits)}")

    circuit = subset_circuit_gates(circuits[0], gate_fraction)
    op_counts = Counter()
    arity_counts = Counter()
    twoq_gate_counts = Counter()
    oneq_count = 0
    twoq_count = 0
    threeplus_count = 0
    measure_count = 0

    for inst in circuit.data:
        name = inst.operation.name
        n_qubits = len(inst.qubits)
        op_counts[name] += 1
        arity_counts[n_qubits] += 1

        if name == "measure":
            measure_count += 1
        if n_qubits == 1 and name != "measure":
            oneq_count += 1
        elif n_qubits == 2:
            twoq_count += 1
            twoq_gate_counts[name] += 1
        elif n_qubits > 2:
            threeplus_count += 1

    return {
        "file": path.name,
        "path": str(path),
        "bytes": path.stat().st_size,
        "circuit_name": circuit.name,
        "num_qubits": circuit.num_qubits,
        "num_clbits": circuit.num_clbits,
        "depth": circuit.depth(),
        "size": circuit.size(),
        "oneq_depth": circuit.depth(
            filter_function=lambda inst: len(inst.qubits) == 1 and inst.operation.name != "measure"
        ),
        "twoq_depth": circuit.depth(filter_function=lambda inst: len(inst.qubits) == 2),
        "oneq_count": oneq_count,
        "twoq_count": twoq_count,
        "threeplus_count": threeplus_count,
        "measure_count": measure_count,
        "op_counts": dict(op_counts),
        "arity_counts": dict(arity_counts),
        "twoq_gate_counts": dict(twoq_gate_counts),
        "count_sx": op_counts.get("sx", 0),
        "count_rz": op_counts.get("rz", 0),
        "count_x": op_counts.get("x", 0),
        "count_cz": op_counts.get("cz", 0),
        "count_measure": op_counts.get("measure", 0),
    }


def percentile(values: pd.Series, q: float) -> float:
    values = values.dropna().sort_values().to_numpy()
    if len(values) == 0:
        return float("nan")
    pos = (len(values) - 1) * q
    lo = math.floor(pos)
    hi = math.ceil(pos)
    if lo == hi:
        return float(values[lo])
    return float(values[lo] * (hi - pos) + values[hi] * (pos - lo))


def describe_metric(df: pd.DataFrame, metric: str) -> dict:
    series = df[metric].dropna()
    min_idx = series.idxmin()
    max_idx = series.idxmax()
    return {
        "metric": metric,
        "mean": float(series.mean()),
        "median": float(series.median()),
        "std": float(series.std(ddof=0)),
        "min": float(series.min()),
        "p05": percentile(series, 0.05),
        "p25": percentile(series, 0.25),
        "p75": percentile(series, 0.75),
        "p95": percentile(series, 0.95),
        "p99": percentile(series, 0.99),
        "max": float(series.max()),
        "min_file": df.loc[min_idx, "file"],
        "max_file": df.loc[max_idx, "file"],
    }

In [18]:
rows: list[dict] = []
errors: list[dict] = []
started = time.time()

print(f"Scanning {len(qpy_files):,} QPY files with {MAX_WORKERS} workers...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(inspect_qpy_file, path): path for path in qpy_files}
    for done, future in enumerate(as_completed(futures), start=1):
        path = futures[future]
        try:
            rows.append(future.result())
        except Exception as exc:
            errors.append({"file": path.name, "error_type": type(exc).__name__, "error": str(exc)})

        if done % PROGRESS_EVERY == 0 or done == len(qpy_files):
            elapsed = time.time() - started
            rate = done / elapsed if elapsed else float("nan")
            print(f"progress={done:,}/{len(qpy_files):,} elapsed_s={elapsed:.1f} rate_files_s={rate:.2f}")

elapsed = time.time() - started
stats_df = pd.DataFrame(rows)
errors_df = pd.DataFrame(errors)

print(f"Loaded circuits: {len(stats_df):,}")
print(f"Errors:          {len(errors_df):,}")
print(f"Elapsed:         {elapsed:.1f}s")

display(stats_df.head())
if len(errors_df):
    display(errors_df.head())

Scanning 7,056 QPY files with 6 workers...
progress=500/7,056 elapsed_s=70.4 rate_files_s=7.10
progress=1,000/7,056 elapsed_s=140.9 rate_files_s=7.10
progress=1,500/7,056 elapsed_s=210.8 rate_files_s=7.12
progress=2,000/7,056 elapsed_s=281.0 rate_files_s=7.12
progress=2,500/7,056 elapsed_s=349.2 rate_files_s=7.16
progress=3,000/7,056 elapsed_s=418.7 rate_files_s=7.17
progress=3,500/7,056 elapsed_s=487.6 rate_files_s=7.18
progress=4,000/7,056 elapsed_s=556.5 rate_files_s=7.19
progress=4,500/7,056 elapsed_s=625.6 rate_files_s=7.19
progress=5,000/7,056 elapsed_s=696.7 rate_files_s=7.18
progress=5,500/7,056 elapsed_s=766.8 rate_files_s=7.17
progress=6,000/7,056 elapsed_s=835.7 rate_files_s=7.18
progress=6,500/7,056 elapsed_s=904.1 rate_files_s=7.19
progress=7,000/7,056 elapsed_s=973.2 rate_files_s=7.19
progress=7,056/7,056 elapsed_s=980.4 rate_files_s=7.20
Loaded circuits: 7,056
Errors:          0
Elapsed:         980.4s


,file,path,bytes,circuit_name,num_qubits,num_clbits,depth,size,oneq_depth,twoq_depth,...,threeplus_count,measure_count,op_counts,arity_counts,twoq_gate_counts,count_sx,count_rz,count_x,count_cz,count_measure
0,000b33f5495fc5d130831e08fec693900a0ae11f64758f...,/Users/noah/repos/Icequake-QRC-/Classical/resu...,1021321,circuit-1602044,156,156,636,19679,463,172,...,0,156,"{'rz': 7716, 'sx': 7887, 'cz': 3451, 'x': 469,...","{1: 16228, 2: 3451}",{'cz': 3451},7887,7716,469,3451,156
1,0022e368359793af850969693cb8edb774e47d1ad4043c...,/Users/noah/repos/Icequake-QRC-/Classical/resu...,571720,circuit-1507955,156,90,228,10975,173,54,...,0,90,"{'sx': 4362, 'rz': 4344, 'cz': 1800, 'x': 379,...","{1: 9175, 2: 1800}",{'cz': 1800},4362,4344,379,1800,90
2,00329f5392105d0179876aee8c467d91cd2c57f832cef4...,/Users/noah/repos/Icequake-QRC-/Classical/resu...,544298,circuit-1656195,156,84,231,10406,176,54,...,0,84,"{'sx': 4053, 'rz': 4240, 'cz': 1680, 'x': 349,...","{1: 8726, 2: 1680}",{'cz': 1680},4053,4240,349,1680,84
3,001b28ec12b3bc69c9068c504e1b9177f6530f2c86576f...,/Users/noah/repos/Icequake-QRC-/Classical/resu...,1003479,circuit-1647022,156,156,777,19386,568,215,...,0,156,"{'rz': 7414, 'sx': 7876, 'cz': 3485, 'x': 455,...","{1: 15901, 2: 3485}",{'cz': 3485},7876,7414,455,3485,156
4,000f3f46445174008364ddedf686241ed745d3e6066155...,/Users/noah/repos/Icequake-QRC-/Classical/resu...,1072385,circuit-1528161,156,156,746,20732,541,205,...,0,156,"{'sx': 8380, 'rz': 7979, 'cz': 3544, 'x': 673,...","{1: 17188, 2: 3544}",{'cz': 3544},8380,7979,673,3544,156


In [19]:
if stats_df.empty:
    raise RuntimeError("No QPY circuits were loaded; cannot summarize stats.")

metrics = [
    "twoq_depth",
    "twoq_count",
    "depth",
    "size",
    "oneq_depth",
    "oneq_count",
    "measure_count",
    "bytes",
]
summary_df = pd.DataFrame(describe_metric(stats_df, metric) for metric in metrics)

op_totals = Counter()
twoq_gate_totals = Counter()
arity_totals = Counter()
for row in stats_df.itertuples(index=False):
    op_totals.update(row.op_counts)
    twoq_gate_totals.update(row.twoq_gate_counts)
    arity_totals.update(row.arity_counts)

op_totals_df = pd.DataFrame(op_totals.most_common(), columns=["operation", "count"])
twoq_gate_totals_df = pd.DataFrame(twoq_gate_totals.most_common(), columns=["operation", "count"])
arity_totals_df = pd.DataFrame(sorted(arity_totals.items()), columns=["operation_qubits", "count"])

overview = {
    "loaded_circuits": int(len(stats_df)),
    "gate_fraction": float(GATE_FRACTION),
    "errors": int(len(errors_df)),
    "total_qpy_size_gb": float(stats_df["bytes"].sum() / 1024**3),
    "qubit_counts": {int(k): int(v) for k, v in stats_df["num_qubits"].value_counts().sort_index().items()},
    "clbit_counts": {int(k): int(v) for k, v in stats_df["num_clbits"].value_counts().sort_index().items()},
    "total_op_counts": {str(k): int(v) for k, v in op_totals.items()},
    "total_twoq_gate_counts": {str(k): int(v) for k, v in twoq_gate_totals.items()},
    "total_arity_counts": {str(k): int(v) for k, v in arity_totals.items()},
    "elapsed_seconds": float(elapsed),
}

display(summary_df)
display(op_totals_df.head(20))
display(twoq_gate_totals_df)
display(arity_totals_df)

,metric,mean,median,std,min,p05,p25,p75,p95,p99,max,min_file,max_file
0,twoq_depth,1.894208e+02,190.0,46.095711,54.0,54.00,170.00,217.00,255.00,293.00,387.0,0022e368359793af850969693cb8edb774e47d1ad4043c...,07e190d30b7254c45bd06544e696e1a269577b0065f10e...
1,twoq_count,3.396512e+03,3512.5,507.228704,480.0,1800.00,3483.00,3539.00,3573.25,3595.45,3638.0,028a6c5ecfbc7bfc507f99eaef382a22b535048c67a9a1...,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...
2,depth,7.113995e+02,715.0,166.248274,211.0,232.00,640.00,811.00,946.00,1088.45,1439.0,0497c1fd5bbd463990262364d72a3bcbcbe178e6229f79...,07e190d30b7254c45bd06544e696e1a269577b0065f10e...
3,size,1.961429e+04,20286.5,2909.052807,2760.0,10919.75,19735.00,20706.00,21032.25,21221.45,21704.0,0ef1f30ea7a9cf875b4ffccc55edbade0a17f65621cb30...,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...
4,oneq_depth,5.230874e+02,527.0,120.883312,156.0,177.00,472.00,595.00,694.00,800.00,1054.0,0497c1fd5bbd463990262364d72a3bcbcbe178e6229f79...,07e190d30b7254c45bd06544e696e1a269577b0065f10e...
5,oneq_count,1.606660e+04,16628.0,2390.068930,2256.0,9029.75,16059.75,17030.00,17335.00,17515.00,17910.0,0ef1f30ea7a9cf875b4ffccc55edbade0a17f65621cb30...,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...
6,measure_count,1.511786e+02,156.0,20.982226,24.0,90.00,156.00,156.00,156.00,156.00,156.0,028a6c5ecfbc7bfc507f99eaef382a22b535048c67a9a1...,000b33f5495fc5d130831e08fec693900a0ae11f64758f...
7,bytes,1.016193e+06,1051145.0,149696.483993,148009.0,568725.25,1022048.50,1071667.75,1090103.25,1100966.30,1125217.0,0ef1f30ea7a9cf875b4ffccc55edbade0a17f65621cb30...,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...


,operation,count
0,sx,55618791
1,rz,53684270
2,cz,23965792
3,x,4062866
4,measure,1066716


,operation,count
0,cz,23965792


,operation_qubits,count
0,1,114432643
1,2,23965792


## Hardware runtime estimate

Uses the cached Z-basis QPY circuits and `FakeFez` gate durations (same scheduling path as `hardware_qrc.py`). For `reuse-z-basis`, each cached circuit corresponds to one transpile unit but three sampler pubs (Z/X/Y); X/Y are assumed to match Z duration.

In [20]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeFez


def format_duration(seconds: float) -> str:
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.1f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.2f}h"


def estimate_circuit_duration_seconds(circuit, target, scheduling_optimization_level: int) -> float | None:
    if not hasattr(circuit, "estimate_duration"):
        return None
    try:
        scheduled_pm = generate_preset_pass_manager(
            target=target,
            optimization_level=scheduling_optimization_level,
            scheduling_method="alap",
        )
        scheduled = scheduled_pm.run(circuit)
    except Exception:
        scheduled = circuit
    try:
        duration = scheduled.estimate_duration(target=target, unit="s")
    except Exception:
        return None
    if duration is None:
        return None
    return float(duration)


def estimate_qpy_duration(
    path: Path,
    target,
    scheduling_optimization_level: int,
    gate_fraction: float = GATE_FRACTION,
) -> dict:
    with path.open("rb") as f:
        circuits = qpy.load(f)
    if len(circuits) != 1:
        raise ValueError(f"{path.name}: expected 1 circuit, got {len(circuits)}")
    circuit = subset_circuit_gates(circuits[0], gate_fraction)
    duration = estimate_circuit_duration_seconds(
        circuit, target, scheduling_optimization_level
    )
    return {"file": path.name, "duration_sec": duration}


backend = FakeFez()
target = backend.target
reset_duration = 0.0
try:
    reset_duration = float(target["reset"][(0,)].duration)
except Exception:
    reset_duration = 0.0

duration_files = list(qpy_files)
if DURATION_MAX_FILES is not None:
    duration_files = duration_files[:DURATION_MAX_FILES]

print(
    f"Estimating per-circuit durations for {len(duration_files):,} QPY files "
    f"on {BACKEND_NAME} (scheduling OL={SCHEDULING_OPTIMIZATION_LEVEL}, "
    f"gate_fraction={GATE_FRACTION:.0%})..."
)
duration_started = time.perf_counter()
duration_rows: list[dict] = []
duration_errors: list[dict] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            estimate_qpy_duration,
            path,
            target,
            SCHEDULING_OPTIMIZATION_LEVEL,
        ): path
        for path in duration_files
    }
    for index, future in enumerate(as_completed(futures), start=1):
        path = futures[future]
        try:
            duration_rows.append(future.result())
        except Exception as exc:
            duration_errors.append({"file": path.name, "error": repr(exc)})
        if index % PROGRESS_EVERY == 0 or index == len(duration_files):
            elapsed = time.perf_counter() - duration_started
            rate = index / elapsed if elapsed > 0 else float("nan")
            print(
                f"progress={index:,}/{len(duration_files):,} "
                f"elapsed_s={elapsed:.1f} rate_files_s={rate:.2f}"
            )

duration_df = pd.DataFrame(duration_rows)
if duration_df.empty:
    raise RuntimeError("No circuit durations were estimated.")

missing_duration = int(duration_df["duration_sec"].isna().sum())
covered = duration_df["duration_sec"].notna()
if missing_duration and covered.any():
    mean_duration = float(duration_df.loc[covered, "duration_sec"].mean())
    duration_df["duration_sec"] = duration_df["duration_sec"].fillna(mean_duration)
    duration_extrapolated = True
elif missing_duration:
    raise RuntimeError("Could not estimate duration for any cached circuit.")
else:
    duration_extrapolated = False

stats_df = stats_df.merge(duration_df, on="file", how="left")

num_transpile_units = int(len(qpy_files))
if len(duration_files) < num_transpile_units:
    mean_z_duration = float(duration_df["duration_sec"].mean())
    z_duration_total = mean_z_duration * num_transpile_units
    duration_scaled_from_sample = True
else:
    z_duration_total = float(duration_df["duration_sec"].sum())
    duration_scaled_from_sample = False
if TRANSPILE_STRATEGY == "independent":
    pubs_per_unit = 1
    num_pubs_total = num_transpile_units
else:
    pubs_per_unit = 3
    num_pubs_total = num_transpile_units * 3

pub_duration_total = z_duration_total * pubs_per_unit
init_seconds_total = reset_duration * num_pubs_total
circuit_seconds_total = (
    pub_duration_total + REP_DELAY_SECONDS * num_pubs_total + init_seconds_total
)
num_executions = num_pubs_total * int(SHOTS)
num_sub_jobs = int(math.ceil(num_pubs_total / max(1, PUBS_PER_JOB)))
quantum_seconds_duration = (
    PER_SUB_JOB_OVERHEAD_SECONDS * num_sub_jobs + circuit_seconds_total * SHOTS
)
quick_formula_seconds = QUICK_SECONDS_PER_EXECUTION * num_executions
quantum_seconds_quick = PER_SUB_JOB_OVERHEAD_SECONDS * num_sub_jobs + quick_formula_seconds

chunks_per_job = max(1, PUBS_PER_JOB // 3)
mean_pub_duration = circuit_seconds_total / max(1, num_pubs_total)
pubs_this_job = chunks_per_job * 3
per_job_seconds_duration = (
    PER_SUB_JOB_OVERHEAD_SECONDS + mean_pub_duration * pubs_this_job * int(SHOTS)
)
per_job_seconds_quick = (
    PER_SUB_JOB_OVERHEAD_SECONDS
    + QUICK_SECONDS_PER_EXECUTION * pubs_this_job * int(SHOTS)
)

runtime_estimate = {
    "backend": BACKEND_NAME,
    "gate_fraction": float(GATE_FRACTION),
    "shots": int(SHOTS),
    "pubs_per_job": int(PUBS_PER_JOB),
    "transpile_strategy": TRANSPILE_STRATEGY,
    "num_transpile_units": num_transpile_units,
    "num_pubs_total": int(num_pubs_total),
    "num_executions": int(num_executions),
    "num_sub_jobs": int(num_sub_jobs),
    "scheduling_optimization_level": int(SCHEDULING_OPTIMIZATION_LEVEL),
    "rep_delay_seconds": float(REP_DELAY_SECONDS),
    "init_qubits_assumed": True,
    "reset_duration_seconds": float(reset_duration),
    "init_seconds_total": float(init_seconds_total),
    "z_circuit_duration_seconds_total": float(z_duration_total),
    "pub_duration_seconds_total": float(pub_duration_total),
    "circuit_seconds_total": float(circuit_seconds_total),
    "duration_scaled_from_sample": bool(duration_scaled_from_sample),
    "duration_sample_size": int(len(duration_files)),
    "duration_extrapolated_missing": bool(duration_extrapolated),
    "missing_duration_circuits": int(missing_duration),
    "duration_coverage_ratio": float(covered.mean()),
    "quantum_seconds_duration_based": float(quantum_seconds_duration),
    "quantum_seconds_quick_formula": float(quantum_seconds_quick),
    "per_job_seconds_duration_based": float(per_job_seconds_duration),
    "per_job_seconds_quick_formula": float(per_job_seconds_quick),
    "duration_elapsed_seconds": float(time.perf_counter() - duration_started),
    "duration_errors": int(len(duration_errors)),
}

duration_summary_df = pd.DataFrame([describe_metric(stats_df, "duration_sec")])

comparison_df = pd.DataFrame(
    [
        {
            "method": "duration_based (FakeFez scheduling)",
            "total_quantum_time": format_duration(quantum_seconds_duration),
            "total_quantum_seconds": quantum_seconds_duration,
            "per_job_time": format_duration(per_job_seconds_duration),
            "per_job_seconds": per_job_seconds_duration,
        },
        {
            "method": "offline_quick_baseline (hardware_qrc.py)",
            "total_quantum_time": format_duration(quantum_seconds_quick),
            "total_quantum_seconds": quantum_seconds_quick,
            "per_job_time": format_duration(per_job_seconds_quick),
            "per_job_seconds": per_job_seconds_quick,
        },
    ]
)

print(
    f"Transpile units: {num_transpile_units:,} | pubs: {num_pubs_total:,} | "
    f"jobs: {num_sub_jobs:,} | shots: {SHOTS:,}"
)
print(
    f"Mean Z-circuit duration: {duration_df['duration_sec'].mean():.4f}s | "
    f"median: {duration_df['duration_sec'].median():.4f}s"
)
if duration_scaled_from_sample:
    print(
        f"Warning: duration estimated on {len(duration_files):,}/{num_transpile_units:,} "
        "cached circuits; scaled total using the sample mean."
    )
if duration_extrapolated:
    print(
        f"Warning: extrapolated {missing_duration:,} missing per-circuit durations "
        "using the mean of covered circuits."
    )
print(
    f"Duration-based TOTAL quantum runtime: {format_duration(quantum_seconds_duration)}"
)
print(
    f"Quick-formula TOTAL quantum runtime:    {format_duration(quantum_seconds_quick)} "
    f"({quantum_seconds_quick / quantum_seconds_duration:.2f}x duration-based)"
)

display(comparison_df)
display(duration_summary_df)

Estimating per-circuit durations for 7,056 QPY files on ibm_fez (scheduling OL=0)...
progress=500/7,056 elapsed_s=150.8 rate_files_s=3.32
progress=1,000/7,056 elapsed_s=303.9 rate_files_s=3.29
progress=1,500/7,056 elapsed_s=457.6 rate_files_s=3.28
progress=2,000/7,056 elapsed_s=608.3 rate_files_s=3.29
progress=2,500/7,056 elapsed_s=756.3 rate_files_s=3.31
progress=3,000/7,056 elapsed_s=905.2 rate_files_s=3.31
progress=3,500/7,056 elapsed_s=1053.7 rate_files_s=3.32
progress=4,000/7,056 elapsed_s=1204.6 rate_files_s=3.32
progress=4,500/7,056 elapsed_s=1355.5 rate_files_s=3.32
progress=5,000/7,056 elapsed_s=1507.8 rate_files_s=3.32
progress=5,500/7,056 elapsed_s=1659.4 rate_files_s=3.31
progress=6,000/7,056 elapsed_s=1808.4 rate_files_s=3.32
progress=6,500/7,056 elapsed_s=1957.9 rate_files_s=3.32
progress=7,000/7,056 elapsed_s=2107.4 rate_files_s=3.32
progress=7,056/7,056 elapsed_s=2122.9 rate_files_s=3.32
Transpile units: 7,056 | pubs: 21,168 | jobs: 43 | shots: 4,096
Mean Z-circuit dura

,method,total_quantum_time,total_quantum_seconds,per_job_time,per_job_seconds
0,duration_based (FakeFez scheduling),6.67h,23996.7468,9.4m,564.526073
1,offline_quick_baseline (hardware_qrc.py),8.45h,30432.4448,11.9m,715.932800


,metric,mean,median,std,min,p05,p25,p75,p95,p99,max,min_file,max_file
0,duration_sec,0.000024,0.000024,0.000005,0.000008,0.000008,0.000022,0.000027,0.000032,0.000036,0.000048,0255b3dd173df5bca0c229494473bdbe6931a6ff2b9897...,64c3033deca7455a99cf00ee837fcc2a8c586a26d68122...


In [ ]:
outlier_rows = []
for metric in metrics:
    min_idx = stats_df[metric].idxmin()
    max_idx = stats_df[metric].idxmax()
    outlier_rows.append({"metric": metric, "kind": "min", **stats_df.loc[min_idx].to_dict()})
    outlier_rows.append({"metric": metric, "kind": "max", **stats_df.loc[max_idx].to_dict()})

outliers_df = pd.DataFrame(outlier_rows)
display(outliers_df[[
    "metric",
    "kind",
    "file",
    "num_qubits",
    "num_clbits",
    "depth",
    "twoq_depth",
    "twoq_count",
    "oneq_depth",
    "oneq_count",
    "measure_count",
    "size",
    "bytes",
]])

,metric,kind,file,num_qubits,num_clbits,depth,twoq_depth,twoq_count,oneq_depth,oneq_count,measure_count,size,bytes
0,twoq_depth,min,0022e368359793af850969693cb8edb774e47d1ad4043c...,156,90,228,54,1800,173,9085,90,10975,571720
1,twoq_depth,max,07e190d30b7254c45bd06544e696e1a269577b0065f10e...,156,156,1439,387,3497,1054,16842,156,20495,1060117
2,twoq_count,min,028a6c5ecfbc7bfc507f99eaef382a22b535048c67a9a1...,156,24,229,54,480,174,2475,24,2979,159832
3,twoq_count,max,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...,156,156,991,278,3638,712,17910,156,21704,1125217
4,depth,min,0497c1fd5bbd463990262364d72a3bcbcbe178e6229f79...,156,84,211,54,1680,156,7901,84,9665,503977
5,depth,max,07e190d30b7254c45bd06544e696e1a269577b0065f10e...,156,156,1439,387,3497,1054,16842,156,20495,1060117
6,size,min,0ef1f30ea7a9cf875b4ffccc55edbade0a17f65621cb30...,156,24,211,54,480,156,2256,24,2760,148009
7,size,max,5ae735da816642a0b86d81b734dd9e5cf4af7c9d2df559...,156,156,991,278,3638,712,17910,156,21704,1125217
8,oneq_depth,min,0497c1fd5bbd463990262364d72a3bcbcbe178e6229f79...,156,84,211,54,1680,156,7901,84,9665,503977
9,oneq_depth,max,07e190d30b7254c45bd06544e696e1a269577b0065f10e...,156,156,1439,387,3497,1054,16842,156,20495,1060117


In [21]:
if SAVE_OUTPUTS:
    stats_csv = OUTPUT_DIR / "qpy_per_circuit_stats.csv"
    summary_csv = OUTPUT_DIR / "qpy_summary_stats.csv"
    op_totals_csv = OUTPUT_DIR / "qpy_operation_totals.csv"
    twoq_totals_csv = OUTPUT_DIR / "qpy_twoq_gate_totals.csv"
    arity_totals_csv = OUTPUT_DIR / "qpy_arity_totals.csv"
    outliers_csv = OUTPUT_DIR / "qpy_outlier_circuits.csv"
    overview_json = OUTPUT_DIR / "qpy_overview.json"
    errors_csv = OUTPUT_DIR / "qpy_load_errors.csv"
    runtime_json = OUTPUT_DIR / "qpy_runtime_estimate.json"
    duration_csv = OUTPUT_DIR / "qpy_duration_per_circuit.csv"
    duration_summary_csv = OUTPUT_DIR / "qpy_duration_summary.csv"
    runtime_comparison_csv = OUTPUT_DIR / "qpy_runtime_comparison.csv"

    # Convert dict-valued columns to JSON strings for CSV readability.
    stats_for_csv = stats_df.copy()
    for column in ["op_counts", "arity_counts", "twoq_gate_counts"]:
        stats_for_csv[column] = stats_for_csv[column].map(lambda value: json.dumps(value, sort_keys=True))

    stats_for_csv.to_csv(stats_csv, index=False)
    summary_df.to_csv(summary_csv, index=False)
    op_totals_df.to_csv(op_totals_csv, index=False)
    twoq_gate_totals_df.to_csv(twoq_totals_csv, index=False)
    arity_totals_df.to_csv(arity_totals_csv, index=False)
    outliers_df.to_csv(outliers_csv, index=False)
    if len(errors_df):
        errors_df.to_csv(errors_csv, index=False)
    overview_with_runtime = {**overview, "runtime_estimate": runtime_estimate}
    with overview_json.open("w") as f:
        json.dump(overview_with_runtime, f, indent=2, sort_keys=True)
    with runtime_json.open("w") as f:
        json.dump(runtime_estimate, f, indent=2, sort_keys=True)
    duration_df.to_csv(duration_csv, index=False)
    duration_summary_df.to_csv(duration_summary_csv, index=False)
    comparison_df.to_csv(runtime_comparison_csv, index=False)

    print("Saved outputs:")
    for path in [
        stats_csv,
        summary_csv,
        op_totals_csv,
        twoq_totals_csv,
        arity_totals_csv,
        outliers_csv,
        overview_json,
        runtime_json,
        duration_csv,
        duration_summary_csv,
        runtime_comparison_csv,
    ]:
        print(f"- {path}")
    if len(errors_df):
        print(f"- {errors_csv}")

Saved outputs:
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_per_circuit_stats.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_summary_stats.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_operation_totals.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_twoq_gate_totals.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_arity_totals.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_outlier_circuits.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_overview.json
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_runtime_estimate.json
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TFI/qpy_transpile_stats/qpy_duration_per_circuit.csv
- /Users/noah/repos/Icequake-QRC-/Classical/results/NN-TF